# Tentativo 19 · 1. L'anonimizzazione

**Perché.** I modelli grandi (Qwen, Mistral) hanno letto il web: se nel testo resta il nome della testata, possono
riconoscerla e usare quello che già sanno della sua affidabilità, invece di leggere l'articolo. Togliamo dal testo
tutto quello che dice **chi** l'ha pubblicato, e niente di quello che dice **che cosa** racconta.

**Il punto di partenza.** Il dataset del tentativo 16 così com'è (226 testate, 25.524 articoli, stessa pulizia con
trafilatura, stessi gruppi). Cambia solo il testo (e il titolo).

**Le regole**, semplici e una per tipo di identificativo, nell'ordine:

| # | che cosa | come si trova | diventa |
|---|---|---|---|
| 1 | la **firma** della testata | la coda che la testata ripete in almeno il 30% dei suoi titoli («… \| The New York Sun») | `[TESTATA]` |
| 2 | email | espressione regolare | `[EMAIL]` |
| 3 | link | http…, www…, domini come `bellingcat.com` | `[URL]` |
| 4 | account | `@nome` | `[ACCOUNT]` |
| 5 | **autori** della testata | il campo autore: nomi che firmano solo in questa testata; nomi di una parola solo dopo «by» | `[AUTORE]` |
| 6 | **nome** della testata | il dominio, scritto attaccato o staccato, con o senza «the» («The American Conservative», «100PercentFedUp»); un «online» in coda non fa parte del nome (spiked-online → spiked) | `[TESTATA]` |

Se il nome è anche una **parola comune** (politics, time, reason…) si toglie solo con l'iniziale maiuscola, così la
parola minuscola resta; tranne quando la testata scrive così il proprio nome, cioè la parola minuscola sta in almeno l'80%
dei suoi articoli (spiked: 100%; il più alto degli altri è time, 56%). Persone, luoghi e fatti di cui parla l'articolo non si toccano. I controlli sono automatici
(sezione 4).

In [1]:
import re

import pandas as pd

import anonimizza as A

pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 80)
OUT = A.Path("Results"); OUT.mkdir(exist_ok=True)
art = pd.read_csv(A.ORIGINE / "articoli.csv.gz")
t = pd.read_csv(A.ORIGINE / "testate.csv", index_col="domain")
print(f"{len(art):,} articoli di {art.domain.nunique()} testate, dal tentativo 16")

25,524 articoli di 226 testate, dal tentativo 16


## 1. Il nome di ogni testata e i nomi che sono parole comuni
Il nome viene dal dominio (`tldextract`, senza rete): la parte prima del suffisso; per le testate su Substack il
sottodominio; senza un «online» in coda. Un nome è una parola comune se compare minuscolo, come parola intera, più di 20 volte negli articoli delle
**altre** testate (fuori da link e account).

In [2]:
nomi = {d: A.nomi_testata(d) for d in sorted(art.domain.unique())}
comuni = A.parole_comuni(nomi, art.text, art.domain)
minuscoli = A.nomi_minuscoli(nomi, comuni, art.text, art.domain)
tab = pd.DataFrame({"nomi": {d: " / ".join(v) for d, v in nomi.items()}}).rename_axis("domain")
tab["parola comune"] = [any((n in comuni or (n.startswith("the") and n[3:] in comuni)) and n not in minuscoli for n in v)
                        for v in nomi.values()]
tab.to_csv(OUT / "nomi.csv")
print(f"{tab['parola comune'].sum()} testate su {len(tab)} hanno un nome che è una parola comune:", ", ".join(sorted(comuni)))
print("parole comuni che una testata usa come proprio nome minuscolo:", ", ".join(sorted(minuscoli)) or "nessuna")
tab[tab["parola comune"]]

18 testate su 226 hanno un nome che è una parola comune: antiwar, bulwark, bureau, dispatch, fee, governing, guardian, mic, nation, national, observer, politics, popular, progressive, racket, reason, spiked, time, week
parole comuni che una testata usa come proprio nome minuscolo: spiked


,nomi,parola comune
domain,,
antiwar.com,antiwar,True
fee.org,fee,True
governing.com,governing,True
mic.com,mic,True
observer.com,observer,True
politics.co.uk,politics,True
popular.info,popular,True
progressive.org,progressive,True
racket.news,racket,True


## 2. Le firme nei titoli e gli autori
La **firma** trova i nomi che dal dominio non si ricavano (cfr.org si firma «Council on Foreign Relations»). Gli **autori**
sono quelli del campo autore che firmano in una sola testata e hanno l'aspetto di un nome di persona; chi firma in più
testate (agenzie, «Staff») non identifica la testata.

In [3]:
firme = A.firme_titoli(art)
autori = A.autori_personali(art)
f = pd.DataFrame({"firma": firme}).rename_axis("domain")
f["nel testo (articoli)"] = [art[art.domain == d].text.astype(str).str.contains(re.escape(x)).sum() for d, x in firme.items()]
f.to_csv(OUT / "firme.csv")
print(f"firme trovate: {len(firme)} testate; autori da togliere: {sum(map(len, autori.values())):,} in {len(autori)} testate")
f

firme trovate: 54 testate; autori da togliere: 4,597 in 198 testate


,firma,nel testo (articoli)
domain,,
100percentfedup.com,by Jack,0
advancingamericanfreedom.com,AAF,40
africanvoiceonline.co.uk,African Voice Newspaper,1
alphanews.org,AlphaNews.org,0
americaoutloud.com,America Out Loud News,0
anthonyblogan.com,Anthony Brian Logan,0
bellingcat.com,bellingcat,0
brookings.edu,Brookings,83
cdm.press,CDM - Human Reporters • Not Machines,0


## 3. L'anonimizzazione

In [4]:
anonimo, conti = A.anonimizza(art, nomi, comuni, minuscoli, autori, firme)
REGOLE = ["firma", "email", "url", "account", "autore", "testata"]
riepilogo = pd.DataFrame({
    "articoli toccati (testo, %)": [(conti[k] > 0).mean() * 100 for k in REGOLE],
    "sostituzioni nel testo": [conti[k].sum() for k in REGOLE],
    "titoli toccati (%)": [(conti[f"titolo_{k}"] > 0).mean() * 100 for k in REGOLE]}, index=REGOLE).round(1)
toccati = (conti[REGOLE].sum(axis=1) + conti[[f"titolo_{k}" for k in REGOLE]].sum(axis=1)) > 0
print(f"articoli con almeno una sostituzione (testo o titolo): {toccati.mean():.1%}")
conti.groupby("domain")[REGOLE].sum().to_csv(OUT / "sostituzioni_per_testata.csv")
riepilogo

articoli con almeno una sostituzione (testo o titolo): 54.6%


,"articoli toccati (testo, %)",sostituzioni nel testo,titoli toccati (%)
firma,6.5,4450,21.6
email,1.3,385,0.0
url,14.5,7860,0.8
account,9.9,7807,0.0
autore,11.3,5235,1.3
testata,17.4,8727,1.0


Qualche esempio, scelto a caso, di che cosa si è tolto (il testo intorno al segnaposto).

In [5]:
for segno in ["[TESTATA]", "[AUTORE]", "[URL]", "[ACCOUNT]"]:
    print("=====", segno)
    for c in A.contesti(anonimo.text, re.compile(re.escape(segno)), quanti=5, larghezza=60, seme=1):
        print("  ·", c)

===== [TESTATA]
  · Expect from Massive Meeting With Xi Jinping (Sources) Watch [TESTATA] every weekday at 11:00 AM Eastern, only on Rumble Premium! 
  · to be identified because of safety concerns, reached out to [TESTATA] with new photos and video of the illegal encampment located
  · d step up the fight for free speech and democracy. Become a [TESTATA] supporter and enjoy unlimited, ad-free access, bonus conten
  · ow to meet our basic operating expenses. If you can support [TESTATA] with a one-time or monthly donation, you will make a signif
  · k of harm being done to Britain is 'just too high'. He told [TESTATA]: "Iranians are smuggling their people into Britain via smal
===== [AUTORE]


  · mation about vaccine safety. Reprinted with permission from [AUTORE]
  · he public gateway for Permit to Purchase information. About [AUTORE] [AUTORE] is a senior editor at [URL] and Liberty Park Press
  · and up for this right, even when we don’t like the outcome. [AUTORE] is a [TESTATA] columnist and author of Human Rights – Illus
  · liance over the safety and lives and voices” of its people. [AUTORE] is a senior staff writer and podcast panellist for [TESTATA
  · ampaign to mask the cost and the consequences,” Murphy told [AUTORE]’s Nicolle Wallace on Thursday. During a congressional deleg
===== [URL]


  ·  we're concerned with," Smith's mother, April Newkirk, told [URL] Photo: GoFundMe Politics The story of Adriana Smith continu
  · zenship fixes came from running records through DHS’s SAVE… [URL] — Eric Daugherty ([ACCOUNT]) October 20, 2026 DHS data expo
  · be a full cover spread like the blood libel was. Evil [URL] [URL] — Saul Sadka ([ACCOUNT]) July 30, 2025 Commentary: This is 
  · nd futile try. So happy he finally licked that nasty flu!!! [URL] — Mark ([ACCOUNT]) October 11, 2026 We'd be VERY interested
  ·  About Puerto Rico This article was originally published at [URL]. Puerto Rico is often seen as a tropical paradise — sunlit 
===== [ACCOUNT]
  · w, dude. You're an idiot and just lost a… — Kelly Clarkson ([ACCOUNT]) October 10, 2026 As of this writing, that post has been se
  · ed to help families buy food. — U.S. Department of Justice ([ACCOUNT]) October 21, 2026 The surge also included a case involving 
  · 026 It's time people wake up to that realization. [ACCOUNT] [A

## 4. I controlli automatici
1. **Il nome e la firma della propria testata** non devono restare come parola intera; si contano anche quelli rimasti
   **dentro** altre parole (refusi come «Quilletteessay», sigle come «AAFF»), con qualche esempio.
2. Per i **nomi che sono parole comuni** resta la parola minuscola, per scelta: si contano a parte.

In [6]:
res = A.residui(anonimo, nomi, comuni, minuscoli, firme)
res.to_csv(OUT / "residui.csv", index=False)
dist = res[~res.parola_comune]
print(f"nomi distintivi e firme ({len(dist)} testate): articoli con un residuo come parola intera {dist.residuo_come_parola.sum()}, "
      f"anche dentro altre parole {dist.residuo_anche_dentro_parole.sum()}")
for d in dist[dist.residuo_anche_dentro_parole > 0].domain:
    g = anonimo[anonimo.domain == d]
    cerca = re.compile("|".join([re.escape(n) for n in nomi[d]] + ([re.escape(firme[d])] if d in firme else [])), re.I)
    print(" ", d, "→", A.contesti(g.text + " " + g.titolo.astype(str), cerca, quanti=2, larghezza=35))
print("\nnomi che sono parole comuni: resta la forma minuscola, per scelta")
res[res.parola_comune]

nomi distintivi e firme (208 testate): articoli con un residuo come parola intera 0, anche dentro altre parole 3
  cf.org → ['artment of Children and Families (DCF) found that many victims are minor', ' notes prior to maturity in 2027,” CFO Lubi Kutua said. Beyond Meat Head']
  quillette.com → ['children. On May 12, I published a Quilletteessay criticizing the Canadian Paed']

nomi che sono parole comuni: resta la forma minuscola, per scelta


,domain,nomi,firma,parola_comune,articoli,residuo_come_parola,residuo_anche_dentro_parole
15,antiwar.com,antiwar,,True,18,0,0
59,fee.org,fee,,True,162,4,43
72,governing.com,governing,,True,158,10,10
109,mic.com,mic,,True,59,0,26
130,observer.com,observer,,True,108,0,0
139,politics.co.uk,politics,Politics.co.uk,True,172,56,57
143,popular.info,popular,,True,190,14,26
148,progressive.org,progressive,,True,20,4,5
152,racket.news,racket,,True,52,0,0
158,reason.com,reason,,True,27,6,10


## 5. Il dataset anonimo

In [7]:
A.DATASET.mkdir(parents=True, exist_ok=True)
colonne = ["article_id", "domain", "titolo", "data", "text", "score", "label", "livello"]   # niente url né autore
anonimo[colonne].to_csv(A.DATASET / "articoli.csv.gz", index=False)
t.drop(columns=["fold"]).to_csv(A.DATASET / "testate.csv")
print(f"scritto {A.DATASET / 'articoli.csv.gz'}: {len(anonimo):,} articoli, {anonimo.domain.nunique()} testate")
print("parole per articolo (mediana): prima", int(art.text.str.split().str.len().median()),
      "dopo", int(anonimo.text.str.split().str.len().median()))

scritto /home/john/Repos/PublisherClassification/Dataset/tentativi/19_DatasetAnonimo/articoli.csv.gz: 25,524 articoli, 226 testate


parole per articolo (mediana): prima 573 dopo 573
